# KhazinaSmart — Feature Engineering

Builds all ML features: temporal, lag, rolling statistics, promotional, and store encoding.

In [7]:
import sys, os
sys.path.insert(0, '..')
import pandas as pd
import plotly.express as px
from src.feature_engineering import build_features, get_feature_columns, get_train_test_split

df = pd.read_csv('../data/processed/walmart_clean.csv', parse_dates=['Date'])
print(f'Loaded clean data: {df.shape}')

Loaded clean data: (29000, 16)


## Build Features

In [8]:
print(f'Shape BEFORE feature engineering: {df.shape}')
df_feat = build_features(df)
print(f'Shape AFTER feature engineering:  {df_feat.shape}')

Shape BEFORE feature engineering: (29000, 16)
[FE] Step 1 — Sorting by Store, Dept, Date | shape: (29000, 16)
[FE] Step 2 — Temporal features | shape: (29000, 16)
[FE] Step 3 — Lag features (Store+Dept grouped) | shape: (29000, 22)
[FE] Step 4 — Rolling stats (Store+Dept grouped) | shape: (29000, 26)
[FE] Step 5 — Promotional features | shape: (29000, 30)
[FE] Step 6 — Store type encoding | shape: (29000, 32)
[FE] Step 7 — Dropping NaN lag rows | shape before: (29000, 35)
[FE] Step 7 done — shape after: (27400, 35)
[FE] Saved features_final.csv — 27400 rows, 35 cols
Shape AFTER feature engineering:  (27400, 35)


## Feature List

In [9]:
feat_cols = get_feature_columns()
print(f'Total features for modeling: {len(feat_cols)}')
for i, col in enumerate(feat_cols, 1):
    print(f'  {i:2d}. {col}')

Total features for modeling: 25
   1. week_of_year
   2. month
   3. quarter
   4. year
   5. is_month_start
   6. is_month_end
   7. IsHoliday
   8. sales_lag_1
   9. sales_lag_2
  10. sales_lag_4
  11. sales_lag_8
  12. rolling_mean_7
  13. rolling_std_7
  14. rolling_mean_30
  15. rolling_std_30
  16. total_markdown
  17. has_markdown
  18. Temperature
  19. Fuel_Price
  20. CPI
  21. Unemployment
  22. store_type_A
  23. store_type_B
  24. store_type_C
  25. Size


## Lag Feature Correlation with Target

In [10]:
lag_cols = ['sales_lag_1', 'sales_lag_2', 'sales_lag_4', 'sales_lag_8',
            'rolling_mean_7', 'rolling_mean_30']
corrs = df_feat[lag_cols + ['Weekly_Sales']].corr()['Weekly_Sales'].drop('Weekly_Sales').sort_values(ascending=False)
fig = px.bar(x=corrs.index, y=corrs.values,
             title='Lag & Rolling Feature Correlation with Weekly_Sales',
             color=corrs.values, color_continuous_scale=['#4a9eff', '#e94560'],
             labels={'x': 'Feature', 'y': 'Pearson Correlation'})
fig.update_layout(height=400, template='plotly_white')
fig.show()

## Train/Test Split

In [11]:
X_train, X_test, y_train, y_test = get_train_test_split(df_feat)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

[Split] Train: 24,400 rows | Test: 3,000 rows
Train: (24400, 25) | Test: (3000, 25)


## Summary

In [12]:
print(f'Feature engineering complete.')
print(f'Total features: {len(get_feature_columns())}. Total rows: {len(df_feat):,}')

Feature engineering complete.
Total features: 25. Total rows: 27,400
